XÂY DỰNG RANDOM FOREST

- Huấn luyện ban đầu

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

thu_muc = r"C:\Users\hauth\OneDrive\Tài liệu\GitHub\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\BÀI TẬP NHÓM HẰNG TUẦN\LAB5 BÀI TOÁN PHÂN LỚP (NHÓM)\THUẬT TOÁN PHÂN LỚP\Thanh\processed"

X_train = pd.read_csv(f"{thu_muc}/X_train.csv")
y_train = pd.read_csv(f"{thu_muc}/y_train.csv")["diabetes"]
X_test = pd.read_csv(f"{thu_muc}/X_test.csv")
y_test = pd.read_csv(f"{thu_muc}/y_test.csv")["diabetes"]

# Huấn luyện Random Forest ban đầu với tham số mặc định
model_rf = RandomForestClassifier(random_state=42, n_jobs=-1)
model_rf.fit(X_train, y_train)

y_pred = model_rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9617784711388455

Classification report:
              precision    recall  f1-score   support

           0       0.97      0.98      0.98     17534
           1       0.82      0.73      0.77      1696

    accuracy                           0.96     19230
   macro avg       0.90      0.86      0.87     19230
weighted avg       0.96      0.96      0.96     19230

Confusion matrix:
[[17264   270]
 [  465  1231]]


- Tinh chỉnh tham số

In [7]:
from sklearn.model_selection import RandomizedSearchCV

# Tìm kiếm bộ tham số tốt nhất bằng RandomizedSearchCV
model_rf.fit(X_train, y_train)

print("Tham số tốt nhất:")
param_distributions = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
}

search_rf = RandomizedSearchCV(
    estimator=model_rf,
    param_distributions=param_distributions,
    n_iter=10,
    scoring="f1_macro",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

search_rf.fit(X_train, y_train)

print("Tham số tốt nhất:")
print(search_rf.best_params_)
print("F1-macro tốt nhất trên tập cross-validation:", search_rf.best_score_)

# Lưu mô hình đã tinh chỉnh để dùng ở các cell tiếp theo
model_rf_tuned = search_rf.best_estimator_
print("F1-macro tốt nhất trên tập cross-validation:", search_rf.best_score_)

# Lưu mô hình đã tinh chỉnh để dùng ở các cell tiếp theo
model_rf_tuned = search_rf.best_estimator_

Tham số tốt nhất:
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Tham số tốt nhất:
{'n_estimators': 100, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'log2', 'max_depth': 30}
F1-macro tốt nhất trên tập cross-validation: 0.9796662307738563
F1-macro tốt nhất trên tập cross-validation: 0.9796662307738563


In [11]:
pred_goc = model_rf.predict(X_test)
pred_tinh_chinh = model_rf_tuned.predict(X_test)

print("Số dự đoán khác nhau:", (pred_goc != pred_tinh_chinh).sum())
print("Accuracy Random Forest ban đầu:", accuracy_score(y_test, pred_goc))
print("Accuracy Random Forest tinh chỉnh:", accuracy_score(y_test, pred_tinh_chinh))
pred_goc = model_rf.predict(X_test)
pred_tinh_chinh = model_rf_tuned.predict(X_test)

print("Số dự đoán khác nhau:", (pred_goc != pred_tinh_chinh).sum())
print("Accuracy Random Forest ban đầu:", accuracy_score(y_test, pred_goc))
print("Accuracy Random Forest tinh chỉnh:", accuracy_score(y_test, pred_tinh_chinh))

print("Số dự đoán khác nhau:",
      (pred_goc != pred_tinh_chinh).sum())
print("Accuracy cây ban đầu:", accuracy_score(y_test, pred_goc))
print("Accuracy cây tinh chỉnh:", accuracy_score(y_test, pred_tinh_chinh))

Số dự đoán khác nhau: 288
Accuracy Random Forest ban đầu: 0.9617784711388455
Accuracy Random Forest tinh chỉnh: 0.9538741549661987
Số dự đoán khác nhau: 288
Accuracy Random Forest ban đầu: 0.9617784711388455
Accuracy Random Forest tinh chỉnh: 0.9538741549661987
Số dự đoán khác nhau: 288
Accuracy cây ban đầu: 0.9617784711388455
Accuracy cây tinh chỉnh: 0.9538741549661987


- Đánh giá mô hình

In [9]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

for ten, model in [
    ("Random Forest ban đầu", model_rf),
    ("Random Forest đã tinh chỉnh", model_rf_tuned),
]:
    y_pred = model.predict(X_test)

    print(f"\n===== {ten} =====")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("F1-macro:", f1_score(y_test, y_pred, average="macro"))
    print("Classification report:")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))


===== Random Forest ban đầu =====
Accuracy: 0.9617784711388455
F1-macro: 0.8746267944974468
Classification report:
              precision    recall  f1-score   support

           0       0.97      0.98      0.98     17534
           1       0.82      0.73      0.77      1696

    accuracy                           0.96     19230
   macro avg       0.90      0.86      0.87     19230
weighted avg       0.96      0.96      0.96     19230

Confusion matrix:
[[17264   270]
 [  465  1231]]

===== Random Forest đã tinh chỉnh =====
Accuracy: 0.9538741549661987
F1-macro: 0.8592593362092643
Classification report:
              precision    recall  f1-score   support

           0       0.98      0.97      0.97     17534
           1       0.73      0.76      0.74      1696

    accuracy                           0.95     19230
   macro avg       0.85      0.87      0.86     19230
weighted avg       0.95      0.95      0.95     19230

Confusion matrix:
[[17055   479]
 [  408  1288]]
